In [7]:
import os
import glob
import math


def calculate_mean_std(values):
    n = len(values)
    if n == 0:
        return 0.0, 0.0
    mean = sum(values) / n
    if n == 1:
        return mean, 0.0
    variance = sum((x - mean) ** 2 for x in values) / (n - 1)
    std = math.sqrt(variance)
    return mean, std


def compute_all_models_table(file_list, output_csv="model_comparison_summary.csv"):
    if not file_list:
        raise FileNotFoundError("No CSV files found matching the provided paths.")

    # Sort files: 1000 first, then 1800; ResNet -> U-Net -> SegNet
    def get_sort_key(path):
        name = os.path.basename(path).lower()
        size_val = 1000 if "1000" in name else 1800
        if "resnet" in name:
            model_val = 1
        elif "unet" in name:
            model_val = 2
        elif "segnet" in name:
            model_val = 3
        else:
            model_val = 99
        return (size_val, model_val)

    sorted_files = sorted(list(set(file_list)), key=get_sort_key)

    metric_patterns = {
        "Accuracy": ["val_accuracy", "val_accu", "val_acc"],
        "Dice": ["val_dice"],
        "IoU": ["val_iou", "val_jaccard"],
        "Precision": ["val_precision", "val_prec"],
        "Recall": ["val_recall", "val_reca", "val_rec", "val_sensitivity"],
        "F1-Score": ["val_f1", "val_f1_score", "val_f1-score"],
    }

    all_rows = []

    for filepath in sorted_files:
        if not os.path.exists(filepath):
            print(f"Warning: File not found -> {filepath}")
            continue

        filename = os.path.splitext(os.path.basename(filepath))[0].lower()

        # Dataset Size label
        if "1800" in filename:
            dataset_size = "Dataset Size: 1800 Images"
        else:
            dataset_size = "Dataset Size: 1000 Images"

        # Model Name label
        if "resnet" in filename:
            model_name = "ResNet"
        elif "unet" in filename:
            model_name = "U-Net"
        elif "segnet" in filename:
            model_name = "SegNet"
        else:
            model_name = filename.capitalize()

        # Read CSV using standard Python file reading (No Pandas/NumPy needed)
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()

        if not lines:
            continue

        header = [h.strip() for h in lines[0].split(",")]
        header_lower = [h.lower() for h in header]

        # Find fold column index to filter out summary rows like 'AVERAGE'
        fold_idx = -1
        for idx, h in enumerate(header_lower):
            if h in ["fold", "kfold", "folds", "split"]:
                fold_idx = idx
                break

        data_rows = []
        for line in lines[1:]:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < len(header):
                continue
            
            # Filter out non-numeric rows if fold column exists
            if fold_idx != -1:
                val = parts[fold_idx]
                try:
                    float(val)
                except ValueError:
                    continue  # Skip rows like 'AVERAGE'
            data_rows.append(parts)

        metrics_summary = {}

        for metric, candidates in metric_patterns.items():
            matched_idx = -1
            for cand in candidates:
                if cand in header_lower:
                    matched_idx = header_lower.index(cand)
                    break

            # Fallback for F1-Score in binary segmentation
            if matched_idx == -1 and metric == "F1-Score":
                for cand in metric_patterns["Dice"]:
                    if cand in header_lower:
                        matched_idx = header_lower.index(cand)
                        break

            if matched_idx != -1:
                vals = []
                for r in data_rows:
                    try:
                        vals.append(float(r[matched_idx]))
                    except ValueError:
                        pass
                
                if vals:
                    mean, std = calculate_mean_std(vals)
                    metrics_summary[metric] = f"{mean:.4f} ± {std:.4f}"
                else:
                    metrics_summary[metric] = "N/A"
            else:
                metrics_summary[metric] = "N/A"

        row = {
            "Dataset Size": dataset_size,
            "Model": model_name,
            "Accuracy": metrics_summary["Accuracy"],
            "Dice": metrics_summary["Dice"],
            "IoU": metrics_summary["IoU"],
            "Precision": metrics_summary["Precision"],
            "Recall": metrics_summary["Recall"],
            "F1-Score": metrics_summary["F1-Score"],
        }
        all_rows.append(row)

    # Save to CSV using standard python file writing
    if output_csv:
        fieldnames = ["Dataset Size", "Model", "Accuracy", "Dice", "IoU", "Precision", "Recall", "F1-Score"]
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            f.write(",".join(fieldnames) + "\n")
            for r in all_rows:
                line_vals = [r[field] for field in fieldnames]
                f.write(",".join(line_vals) + "\n")
        print(f"\n[Success] Table saved successfully to: {output_csv}")

    return all_rows


# ========================================================
# Execution (Using your exact folder path E:\BIOSE\cvs)
# ========================================================
if __name__ == "__main__":
    folder_path = r"E:\BIOSE\cvs"
    
    manual_files_list = [
        os.path.join(folder_path, "resnet_1000.csv"),
        os.path.join(folder_path, "unet_1000.csv"),
        os.path.join(folder_path, "segnet_1000.csv"),
        os.path.join(folder_path, "resnet_1800.csv"),
        os.path.join(folder_path, "unet_1800.csv"),
        os.path.join(folder_path, "segnet_1800.csv")
    ]

    output_file_path = os.path.join(folder_path, "model_comparison_summary.csv")
    summary_rows = compute_all_models_table(manual_files_list, output_csv=output_file_path)

    print("=" * 115)
    print("Table 2: Performance comparison of U-Net, SegNet, and ResNet (mean ± std across folds).")
    print("=" * 115)
    # Print formatted table nicely in console
    header_str = f"{'Dataset Size':<25} | {'Model':<10} | {'Accuracy':<18} | {'Dice':<18} | {'IoU':<18} | {'Precision':<18} | {'Recall':<18} | {'F1-Score':<18}"
    print(header_str)
    print("-" * len(header_str))
    for r in summary_rows:
        print(f"{r['Dataset Size']:<25} | {r['Model']:<10} | {r['Accuracy']:<18} | {r['Dice']:<18} | {r['IoU']:<18} | {r['Precision']:<18} | {r['Recall']:<18} | {r['F1-Score']:<18}")
    print("=" * 115)


[Success] Table saved successfully to: E:\BIOSE\cvs\model_comparison_summary.csv
Table 2: Performance comparison of U-Net, SegNet, and ResNet (mean ± std across folds).
Dataset Size              | Model      | Accuracy           | Dice               | IoU                | Precision          | Recall             | F1-Score          
--------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dataset Size: 1000 Images | ResNet     | 0.8075 ± 0.0634    | 0.8122 ± 0.0756    | 0.6891 ± 0.1049    | 0.7840 ± 0.0714    | 0.8553 ± 0.1293    | 0.8122 ± 0.0756   
Dataset Size: 1000 Images | U-Net      | 0.9230 ± 0.0163    | 0.7130 ± 0.0529    | 0.6102 ± 0.0594    | 0.7473 ± 0.0770    | 0.7741 ± 0.0485    | 0.7130 ± 0.0529   
Dataset Size: 1000 Images | SegNet     | 0.8751 ± 0.0070    | 0.4296 ± 0.0343    | 0.3091 ± 0.0287    | 0.5420 ± 0.0417    | 0.4044 ± 0.0336    | 0.4296 ± 0.0343   
Datas